# Assignment: Feature Engineering with the Heart Disease Dataset


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

In [ ]:
# Load the dataset
heart_df = pd.read_csv("heart.csv")

In [ ]:
# Display basic information about the dataset
heart_df.info()

Output:     RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   age       1025 non-null   int64
 1   sex       1025 non-null   int64
 2   cp        1025 non-null   int64
 3   trestbps  1025 non-null   int64
 4   chol      1025 non-null   int64
 5   fbs       1025 non-null   int64
 6   restecg   1025 non-null   int64
 7   thalach   1025 non-null   int64
 8   exang     1025 non-null   int64
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64
 11  ca        1025 non-null   int64
 12  thal      1025 non-null   int64
 13  target    1025 non-null   int64
dtypes: float64(1), int64(13)
memory usage: 112.2 KB

In [ ]:
# Check the shape of the dataset (rows, columns)
heart_df.shape

Output: (1025, 14)

The dataset contains 1025 rows and 14 columns, meaning there are 303 patient records and 14 variables describing their medical information.

In [ ]:
# Display the first 5 rows
heart_df.head()

In [ ]:
# Identify categorical columns in the dataframe
categorical_cols = heart_df.select_dtypes(include='object').columns
categorical_cols

Output: Index([], dtype='str')

The DataFrame currently has no columns with datatype object.

In [ ]:
# Identify numerical columns in the dataframe
numerical_cols = heart_df.select_dtypes(include='number').columns
numerical_cols

Output:  Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'],
      dtype='str')

 These columns are identified as numerical columns.

In [ ]:
# Check the data types of all columns
heart_df.dtypes

In [ ]:
# Check if there are missing values in the dataset
heart_df.isnull().sum()


age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

There are no missing values in the dataset, so no imputation is required.

In [ ]:
# Create age groups using binning
heart_df['age_group'] = pd.cut(
    heart_df['age'],
    [30, 40, 50, 60, 70],
    labels=['30-40', '40-50', '50-60', '60-70']
)

Age is grouped into intervals (30–40, 40–50, etc.) to analyze patterns across age groups rather than individual ages.

In [ ]:
# Create cholesterol level categories
heart_df['chol_bin'] = pd.cut(
    heart_df["chol"],
    [0, 200, 240, np.inf],
    labels=["Normal", "Borderline", "High"]
)

Cholesterol levels have been categorized into Normal, Borderline, and High to simplify analysis and improve model interpretability.

In [ ]:
# Check frequency of cholesterol bins
heart_df['chol_bin'].value_counts()

Output: chol_bin

High          503

Borderline    350

Normal        172

Name: count, dtype: int64

In [ ]:
# Convert newly created categorical columns to object type
heart_df[['chol_bin', 'age_group']] = heart_df[['chol_bin', 'age_group']].astype('object')

In [ ]:
# Check updated data types
heart_df.dtypes

age            int64
sex            int64
cp             int64
trestbps       int64
chol           int64
fbs            int64
restecg        int64
thalach        int64
exang          int64
oldpeak      float64
slope          int64
ca             int64
thal           int64
target         int64
age_group     object
chol_bin      object
dtype: object

In [ ]:
# Create new feature: pulse_age_ratio
heart_df["pulse_age_ratio"] = heart_df["thalach"] / (heart_df["age"] + 1)

A new feature called pulse_age_ratio was created to represent the relationship between heart rate (thalach) and age.

In [ ]:
# Apply one-hot encoding to categorical features
## Removes the first category to avoid multicollinearity
heart_df = pd.get_dummies(
    heart_df,
    columns=["sex","cp","fbs","restecg","exang","slope","thal","chol_bin","age_group"],
    drop_first=True
)

Categorical variables were converted into dummy variables using one-hot encoding so they can be used by machine learning algorithms. The first category was dropped (drop_first=True) to avoid multicollinearity.

In [ ]:
# Check data types after encoding
print(heart_df.dtypes)

In [ ]:
# Convert boolean columns created by get_dummies into integers
bool_cols = heart_df.select_dtypes(include='bool').columns
heart_df[bool_cols] = heart_df[bool_cols].astype(int)

Boolean columns generated from one-hot encoding were converted into integer format to ensure compatibility with machine learning models.

In [ ]:
## Confirm the changes
print(heart_df.dtypes)

In [ ]:
# Standardization
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Standardize selected numerical features
heart_df[['age', 'chol', 'trestbps', 'thalach', 'oldpeak', 'pulse_age_ratio']] = scaler.fit_transform(
    heart_df[['age', 'chol', 'trestbps', 'thalach', 'oldpeak', 'pulse_age_ratio']]
)

After applying standardization, some feature values become negative or positive because the data is centered around the mean. Values below the mean become negative, while values above the mean become positive

In [ ]:
# Export the cleaned dataset
heart_df.to_csv("heart_disease_cleaned.csv", index=False)

The final cleaned dataset is exported as heart_disease_cleaned.csv and is now ready for machine learning model training.